#### Import the required packages

In [2]:
import pandas as pd
import numpy as np
import re
# Train-test split
from  sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate
#TF-IDF Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
#models
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
#evaluation of the model
from sklearn.metrics import (accuracy_score,classification_report,confusion_matrix,ConfusionMatrixDisplay)
#visualization 
import matplotlib.pyplot as plt
# nltk
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV


nltk.download("stopwords")

[nltk_data] Error loading stopwords: <urlopen error pathsec.urlopen:
[nltk_data]     no validated address for host
[nltk_data]     'raw.githubusercontent.com'; refusing to connect by
[nltk_data]     unvalidated hostname>


False

#### Load the dataset

In [3]:
# Load the cleaned dataset and clinical stop words
df = pd.read_csv("../data/top_mtsamples.csv")
df.shape

(3104, 6)

In [4]:
# Load custom clinical stop words
with open("../data/clinical-stopwords.txt") as f:
    clinical_stopwords={
        line.strip().lower()
        for line in f
        if line.strip() and not line.startswith("#")
    }
# Load NLTK English stop words
english_stopwords = set(stopwords.words("english"))
# Combine English and clinical stop words
all_stopwords = english_stopwords.union(clinical_stopwords)

#### Prepreocessing

In [5]:
#create a prepreocessing function
def preprocess_text(text):
    """
    Clean and preprocess clinical text for TF-IDF.
    """

    # Handle missing values
    if pd.isna(text):
        return ""

    # Convert text to lowercase
    text = text.lower()

    # Remove numbers and punctuation (keep only letters and spaces)
    text = re.sub(r"[^a-z\s]", " ", text)

    # Split text into individual words
    tokens = word_tokenize(text)

    # Remove stop words and lemmatize the remaining words
    lemmatizer = WordNetLemmatizer()
    cleaned_tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in clinical_stopwords
    ]

    # Join the cleaned words back into a sentence
    return " ".join(cleaned_tokens)

In [6]:
# Apply preprocessing to all transcriptions
df["cleaned_text"] = df["transcription"].apply(preprocess_text)
df.cleaned_text

0       mode left atrial enlargement left atrial diame...
1       left ventricular cavity size wall thickness no...
2       echocardiogram multiple view heart great vesse...
3       description normal cardiac chamber size normal...
4       study mild aortic stenosis widely calcified mi...
                              ...                        
3099    indication chest pain type test adenosine nucl...
3100    chief complaint chest pain history present ill...
3101    history present illness year woman following a...
3102    history present illness mr abc year gentleman ...
3103    reason consultation abnormal echocardiogram fi...
Name: cleaned_text, Length: 3104, dtype: str

#### Define the features and labels

In [7]:
# Input features
X = df["cleaned_text"]
# Target labels
y = df["medical_specialty_clean"]

#### Split the data into training and testing sets

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.3,random_state=42)
#print(y_train.value_counts(normalize=True))


#### Create the TF-IDF Vectorizer | Training | Evaluating

In [12]:
# Create TF-IDF vectorizer
tfidf = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

# Learn the vocabulary from the training data
X_train_tfidf = tfidf.fit_transform(X_train)

# Transform the test data using the same vocabulary
X_test_tfidf = tfidf.transform(X_test)

# Create the Linear SVC model
svc_model = LinearSVC(
    class_weight="balanced",
    random_state=42
)

# 5-fold Cross Validation
scores = cross_validate(
    svc_model,
    X_train_tfidf,
    y_train,
    cv=5,
    scoring="f1_macro"
)

print("Fold Scores:", scores["test_score"])
print(f"Mean Macro F1: {np.mean(scores['test_score']):.4f}")

# Train the model
svc_model.fit(X_train_tfidf, y_train)

# Predict on the TEST set
y_pred_svc = svc_model.predict(X_test_tfidf)

# Evaluate
accuracy = accuracy_score(y_test, y_pred_svc)
print(f"Accuracy: {accuracy:.4f}")

print(classification_report(y_test, y_pred_svc))

Fold Scores: [0.39564242 0.42443969 0.37607741 0.3541731  0.39637706]
Mean Macro F1: 0.3893
Accuracy: 0.3873
                            precision    recall  f1-score   support

Cardiovascular / Pulmonary       0.37      0.46      0.41       113
          Gastroenterology       0.30      0.25      0.27        71
          General Medicine       0.76      0.82      0.79        82
                 Neurology       0.39      0.56      0.46        55
   Obstetrics / Gynecology       0.27      0.38      0.32        39
                Orthopedic       0.28      0.34      0.31        96
                 Radiology       0.25      0.19      0.22        85
                   Surgery       0.40      0.33      0.36       346
                   Urology       0.36      0.36      0.36        45

                  accuracy                           0.39       932
                 macro avg       0.38      0.41      0.39       932
              weighted avg       0.39      0.39      0.38       932



#### Pipeline with Linear SVC model and Hyperparameter tuning

In [ ]:
# Create a pipeline
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("svc", LinearSVC(class_weight="balanced", random_state=42))
])


# Define the hyperparameter grid
param_grid = {
    "tfidf__max_features": [5000,20000],
    "tfidf__min_df": [1,2, 3,5],
    "tfidf__max_df": [0.85,0.90,0.95],
    "tfidf__ngram_range": [(1,1), (1,2),(1,3)],
    "tfidf__norm":["l2"],
    "tfidf__sublinear_tf":[True,False],
    "svc__C": [0.01,0.1, 1, 5,10]
}

# Grid Search
grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1,
    verbose=0
)

# Train the pipeline
grid.fit(X_train, y_train)

# Best results
print("Best Parameters:")
print(grid.best_params_)

print("\nBest Cross-Validation Macro F1:")
print(grid.best_score_)

#get the best model
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")
print(classification_report(y_test, y_pred))

# Predict the medical specialties of the test set
y_pred_svc = svc_model.predict(X_test_tfidf)
# View the first 10 predictions
comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred_svc
})

#### Confusion Matrix

In [ ]:
# Create confusion matrix
cm = confusion_matrix(y_test, y_pred_svc)
fig, ax = plt.subplots(figsize=(10,10))
im = ax.imshow(cm, interpolation='nearest', cmap="Oranges")
cbar = plt.colorbar(im)
cbar.ax.tick_params(labelsize=11)
classes = svc_model.classes_
ax.set(
    xticks=np.arange(len(classes)),
    yticks=np.arange(len(classes)),
    xticklabels=classes,
    yticklabels=classes,
    xlabel='Predicted Label',
    ylabel='True Label',
    title='Confusion Matrix - Confusion Matrix - Linear SVC'
)
plt.setp(ax.get_xticklabels(),
         rotation=45,
         ha="right",
         rotation_mode="anchor")
threshold = cm.max() / 2
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(
            j,
            i,
            format(cm[i, j], "d"),
            ha="center",
            va="center",
            fontsize=10,
            color="white" if cm[i, j] > threshold else "black"
        )

plt.tight_layout()
plt.show()

#### Create the multinormial Logistic Regression Model

In [ ]:
# Create the Logistic Regression model
lr_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)


In [ ]:
#calculate Macro F1  for comparison
scores = cross_validate(
    lr_model,
    X_train_tfidf,
    y_train,
    cv=5,
    scoring="f1_macro"
)

print("Fold scores:", scores)
#print("LR Mean Macro F1:", scores)
print("LR Mean Macro F1:",  np.mean(scores["test_score"]))

Fold scores: {'fit_time': array([2.54447269, 2.78220487, 2.91758156, 2.62903619, 3.2508502 ]), 'score_time': array([0.00920558, 0.01446342, 0.01367879, 0.00565004, 0.00774026]), 'test_score': array([0.49419388, 0.49415833, 0.49194428, 0.48283931, 0.51625472])}
LR Mean Macro F1:


#### Train the Model

In [ ]:
# Train the model
lr_model.fit(X_train_tfidf, y_train)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default 

#### Make Predictions

In [ ]:
# Predict the test set
y_pred_lr = lr_model.predict(X_test_tfidf)

#### Evaluate Accuracy and Classification Report

In [ ]:
accuracy_lr = accuracy_score(y_test, y_pred_lr)
print(f"Accuracy: {accuracy_lr:.4f}")
print(classification_report(y_test, y_pred_lr))


Accuracy: 0.5268
                            precision    recall  f1-score   support

Cardiovascular / Pulmonary       0.48      0.67      0.56       113
          Gastroenterology       0.53      0.66      0.59        71
          General Medicine       0.70      0.89      0.78        82
                 Neurology       0.43      0.64      0.51        55
   Obstetrics / Gynecology       0.42      0.77      0.55        39
                Orthopedic       0.45      0.71      0.55        96
                 Radiology       0.31      0.24      0.27        85
                   Surgery       0.70      0.31      0.43       346
                   Urology       0.56      0.76      0.64        45

                  accuracy                           0.53       932
                 macro avg       0.51      0.63      0.54       932
              weighted avg       0.57      0.53      0.51       932



#### Confusion Matrix

In [ ]:


# # Create confusion matrix
# cm = confusion_matrix(y_test, y_pred_lr)
# fig, ax = plt.subplots(figsize=(10,10))
# im = ax.imshow(cm, interpolation='nearest', cmap="Oranges")
# cbar = plt.colorbar(im)
# cbar.ax.tick_params(labelsize=11)
# classes = lr_model.classes_
# ax.set(
#     xticks=np.arange(len(classes)),
#     yticks=np.arange(len(classes)),
#     xticklabels=classes,
#     yticklabels=classes,
#     xlabel='Predicted Label',
#     ylabel='True Label',
#     title='Confusion Matrix - Multinomial Logistic Regression'
# )
# plt.setp(ax.get_xticklabels(),
#          rotation=45,
#          ha="right",
#          rotation_mode="anchor")
# threshold = cm.max() / 2
# for i in range(cm.shape[0]):
#     for j in range(cm.shape[1]):
#         ax.text(
#             j,
#             i,
#             format(cm[i, j], "d"),
#             ha="center",
#             va="center",
#             fontsize=10,
#             color="white" if cm[i, j] > threshold else "black"
#         )

# plt.tight_layout()
# # plt.show()

In [ ]:
# Create a pipeline
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("lr", LogisticRegression(class_weight="balanced", max_iter=1000,random_state=42))
])

# Define the hyperparameter grid
param_grid = {
    "tfidf__max_features": [5000,20000],
    "tfidf__min_df": [1,2, 3,5],
    "tfidf__max_df": [0.85,0.90,0.95],
    "tfidf__ngram_range": [(1,1), (1,2),(1,3)],
    "tfidf__norm":["l2"],
    "tfidf__sublinear_tf":[True,False],
    "lr__C": [0.01,0.1, 1, 5,10]
}

# Grid Search
grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1,
    verbose=0
)

# Train the pipeline
grid.fit(X_train, y_train)

# Best results
print("Best Parameters:")
print(grid.best_params_)

print("\nBest Cross-Validation Macro F1:")
print(grid.best_score_)

#get the best model
best_model = grid.best_estimator_
y_pred_lr = best_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred_lr)

print("\nModel with Hyperparameters")
print("-----------------------------------------")
print(f"Accuracy: {accuracy:.2f}")
print(classification_report(y_test, y_pred_lr))

Best Parameters:
{'lr__C': 0.1, 'tfidf__max_df': 0.85, 'tfidf__max_features': 5000, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 1), 'tfidf__norm': 'l2', 'tfidf__sublinear_tf': True}

Best Cross-Validation Macro F1:
0.529357610312064

Model with Hyperparameters
-----------------------------------------
Accuracy: 0.54
                            precision    recall  f1-score   support

Cardiovascular / Pulmonary       0.48      0.53      0.50       113
          Gastroenterology       0.51      0.58      0.54        71
          General Medicine       0.54      0.95      0.69        82
                 Neurology       0.50      0.64      0.56        55
   Obstetrics / Gynecology       0.40      0.64      0.49        39
                Orthopedic       0.47      0.76      0.58        96
                 Radiology       0.48      0.35      0.41        85
                   Surgery       0.76      0.36      0.49       346
                   Urology       0.56      0.78      0.65        45

In [38]:
# print("=" * 100)
# print("LINEAR SVC - BEFORE HYPERPARAMETER TUNING".center(100))
# print("=" * 100)

# print("\nCross-Validation Results")
# print("-" * 100)
# print("Fold Scores:", scores["test_score"])
# print(f"Mean Macro F1 : {np.mean(scores['test_score']):.4f}")

# print("\nTest Set Performance")
# print("-" * 100)
# print(f"Accuracy      : {accuracy_score(y_test, y_pred_svc):.4f}")
# print("\nClassification Report")
# print(classification_report(y_test, y_pred_svc))


# print("\n" + "=" * 100)
# print("LINEAR SVC - AFTER HYPERPARAMETER TUNING".center(100))
# print("=" * 100)

# print("\nBest Hyperparameters")
# print("-" * 100)
# for param, value in grid.best_params_.items():
#     print(f"{param:<25}: {value}")

# print(f"\nBest CV Macro F1 : {grid.best_score_:.4f}")

# #best_model = grid.best_estimator_
# #y_pred = best_model.predict(X_test)

# print("\nTest Set Performance")
# print("-" * 100)
# print(f"Accuracy      : {accuracy_score(y_test, y_pred):.4f}")

# print("\nClassification Report")
# print(classification_report(y_test, y_pred))


# print("\n\n" + "=" * 100)
# print("LOGISTIC REGRESSION - BEFORE HYPERPARAMETER TUNING".center(100))
# print("=" * 100)

# print("\nCross-Validation Results")
# print("-" * 100)
# print("Fold Scores:", scores["test_score"])
# print(f"Mean Macro F1 : {np.mean(scores['test_score']):.4f}")

# print("\nTest Set Performance")
# print("-" * 100)
# print(f"Accuracy      : {accuracy_lr:.4f}")

# print("\nClassification Report")
# print(classification_report(y_test, y_pred_lr))


# print("\n" + "=" * 100)
# print("LOGISTIC REGRESSION - AFTER HYPERPARAMETER TUNING".center(100))
# print("=" * 100)

# print("\nBest Hyperparameters")
# print("-" * 100)
# for param, value in grid.best_params_.items():
#     print(f"{param:<25}: {value}")

# print(f"\nBest CV Macro F1 : {grid.best_score_:.4f}")

# best_model = grid.best_estimator_
# y_pred_lr = best_model.predict(X_test)

# print("\nTest Set Performance")
# print("-" * 100)
# print(f"Accuracy      : {accuracy_score(y_test, y_pred_lr):.4f}")

# print("\nClassification Report")
# print(classification_report(y_test, y_pred_lr))

# print("\n" + "=" * 100)
# print("END OF MODEL COMPARISON".center(100))
# print("=" * 100)